In [1]:
# Parse CC-Canto and CE-Dict
from pathlib import Path 
import re 

data = {
    "cccanto": [],
    "cedict": []
}

for f in Path("data").iterdir():
    parse = False
    if f.name.startswith("cccanto"):
        parse = True 
        data_type = "cccanto"
    elif f.name.startswith("cedict"):
        parse = True 
        data_type = "cedict"
    
    if parse:
        for line in f.read_text().splitlines():
            if not line.startswith("#"):
                # Split on the first space to get traditional, then the rest
                traditional, _, rest = line.partition(' ')
                
                # Split the rest on space to get simplified, then remainder
                simplified, _, rest = rest.partition(' ')

                pinyin = re.search(r"\[(.*?)\]", rest).group(1)

                definition = re.search(r"/(.*)\/", rest).group(1)

                jyutping_match = re.search(r"{(.*?)}", rest)
                jyutping = ""
                if jyutping_match is not None:
                    jyutping = jyutping_match.group(1)
                
                # NOTE: CE-Dict contains Cantonese entries
                if data_type == "cedict" and "(Cantonese)" in definition:
                    continue 

                row = {
                    "traditional": traditional,
                    "simplified": simplified,
                    "pinyin": pinyin,
                    "jyutping": jyutping,
                    "definition": definition
                }
                data[data_type].append(row)

In [2]:
data["cccanto"][0]

{'traditional': '一件還一件',
 'simplified': '一件还一件',
 'pinyin': 'yi1 jian4 hai2 yi2 jian4',
 'jyutping': 'jat1 gin6 waan4 jat1 gin6',
 'definition': "a different matter/don't mix another incident with this one"}

In [3]:
data["cedict"][0]


{'traditional': '%',
 'simplified': '%',
 'pinyin': 'pa1',
 'jyutping': '',
 'definition': 'percent (Tw)'}

In [4]:
# Find entries that are in CC-Canto but not in CE-Dict
import polars as pl 

canto_df = pl.DataFrame(data["cccanto"]).unique(subset=["traditional", "definition"])
mando_df = pl.DataFrame(data["cedict"]).unique(subset=["traditional", "definition"])
canto_words = set(canto_df.select("traditional").to_series())
mando_words = set(mando_df.select("traditional").to_series())

unique_canto_words = canto_words - mando_words 
print(f"Cantonese words: {len(canto_words)}")
print(f"Mandarin words: {len(mando_words)}")
print(f"Unique Cantonese words: {len(unique_canto_words)}")

Cantonese words: 29698
Mandarin words: 121286
Unique Cantonese words: 21141


In [5]:
homonym_canto_words = set()

for unique_word in unique_canto_words:
    # Filter for the specific word
    word_df = canto_df.filter(pl.col("traditional") == unique_word)
    
    # Check if there are multiple entries
    if len(word_df) > 1:
        homonym_canto_words.add(unique_word)

In [6]:
print(sorted(homonym_canto_words)[:10])

['㕵', '㗎', '㗎啦', '㨢', '㷫', '一字咁淺', '三味', '上莊', '仱', '俹']


In [7]:
# Now take all the different definitions and use them as senses 
# Format in a spreadsheet for review

homonym_canto_df = canto_df.filter(pl.col("traditional")
                                   .is_in(homonym_canto_words)) \
                                   .sort(by="traditional")
result = homonym_canto_df.write_csv(
    "./output/dictionary_senses.csv"
)

In [8]:
import json 

with open("./output/cedict.json", "w") as outfile:
    json.dump(data["cedict"], outfile, indent=4)

with open("./output/cccanto.json", "w") as outfile:
    json.dump(data["cccanto"], outfile, indent=4)